# **Importing the needed libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# **Importing the DataSets**

In [ ]:
order_df = pd.read_json(open("orders_dataset.json", "r", encoding="utf8"))
order_df.head()

In [ ]:
review_df = pd.read_json(open("order_reviews_dataset.json", "r", encoding="utf8"))
review_df.head()

In [ ]:
product_df = pd.read_json(open("products_dataset.json", "r", encoding="utf8"))
product_df.head()

In [ ]:
item_df = pd.read_json(open("order_items_dataset.json", "r", encoding="utf8"))
item_df.head()

In [ ]:
cat_trans = pd.read_json(open("product_category_name_translation.json", "r", encoding="utf8"))
cat_trans.head()

#**exploring the Order Dataset**

In [ ]:
#checking for null values
order_df.isnull().sum()

In [ ]:
#checking for duplicates
order_df.duplicated().sum()

In [ ]:
#checking the types of the columns
order_df.dtypes

In [ ]:
#dropping null values
order_df.dropna(subset=["order_approved_at"], inplace=True)

In [ ]:
order_df.isnull().sum()

#**exploring the Order review Dataset**

In [ ]:
#checking for null values
review_df.isnull().sum()

In [ ]:
#checking for duplicates
review_df.duplicated().sum()

In [ ]:
#checking the types of the columns
review_df.dtypes

# **exploring the order items data**

In [ ]:
#checking for null values
item_df.isnull().sum()

In [ ]:
#checking for duplicates
item_df.duplicated().sum()

In [ ]:
#checking the types of the columns
item_df.dtypes

# **exploring the products data**


In [ ]:
#checking for null values
product_df.isnull().sum()

In [ ]:
#checking for duplicates
product_df.duplicated().sum()

In [ ]:
#checking the types of the columns
product_df.dtypes

# **exploring the product_category_name_translation data**


In [ ]:
#checking for null values
cat_trans.isnull().sum()

In [ ]:
#checking for duplicates
cat_trans.duplicated().sum()

In [ ]:
#checking the types of the columns
cat_trans.dtypes

# **merging datasets**

In [ ]:
#merging the orders and review data
order_reviews = pd.merge(order_df, review_df, on="order_id", how="inner")
#merging with items data
order_reviews_items = pd.merge(order_reviews, item_df, on="order_id", how="inner")
#merging with product data
merged = pd.merge(order_reviews_items, product_df, on="product_id", how="left")

In [ ]:
merged.head()

In [ ]:
#changing the name of the product_category_name column
print(cat_trans.columns.tolist())
cat_trans.columns = cat_trans.columns.str.strip().str.replace('\ufeff','', regex=True)
print(cat_trans.columns.tolist())

In [ ]:
#creating a dictonary to translate the product category name to english
trans_dict = dict(zip(cat_trans["product_category_name"],
                      cat_trans["product_category_name_english"]))

In [ ]:
#maping between the category name in brazilian and english
merged["product_category_name_english"] = merged["product_category_name"].map(trans_dict)
#dropping the brazilian category column
merged.drop(columns=["product_category_name"], inplace=True)
merged.head()

# **exploring the merged data**

In [ ]:
#checking for null values
merged.isnull().sum()

In [ ]:
#checking the types
merged.dtypes

In [ ]:
#dropping the null values
merged.dropna(inplace=True)

In [ ]:
#checking for duplicates
merged.duplicated().sum()

# **Feature engneering**

In [ ]:
#Removing the time in hours because we only need days
columns = ["order_estimated_delivery_date", "order_delivered_customer_date"]
merged[columns] = (merged[columns].apply(pd.to_datetime, errors="coerce"))
merged[columns] = (merged[columns].apply(lambda x: x.dt.normalize()))

In [ ]:
#calculating the num of delay days
merged["delay_num_days"] = (merged["order_delivered_customer_date"] - merged["order_estimated_delivery_date"]).dt.days

In [ ]:
# creating a new column for delay category
def classify_delay(x):
    if x < 0: return "Early"
    if x == 0: return "Ontime"
    return "Delayed"
merged["delay_category"] = merged["delay_num_days"].apply(classify_delay)

In [ ]:
# creating a new column for satisfaction category
def classify_satisfaction(x):
    x = float(x)
    if x <= 2:
        return "not satisfied"
    elif x == 3:
        return "satisfied"
    elif x >= 4:
        return "extremely satisfied"
merged["satisfaction_category"] = merged["review_score"].apply(classify_satisfaction)

In [ ]:
# aggregation using price and num of items
order_items_aggregate = (
    # grouping all items by order_id
    item_df.groupby("order_id")
    # counting how many items in the order
    .agg(num_items=("order_item_id", "count"),
    # summing  all item prices to find total order value
         total_price=("price", "sum"))
    # reset index so order_id stays a column
    .reset_index()
)
# add the columns to merged dataset
merged = merged.merge(order_items_aggregate, on="order_id", how="left")

In [ ]:
# Aggregation using freight
freight_aggregate = (
    # grouping all items by order_id
    item_df.groupby("order_id")
    # summing shipping cost for all items in the same order
    .agg(total_freight=("freight_value","sum"))
    # reset index so order_id stays a column
    .reset_index()
)

# add the columns to merged dataset
merged = merged.merge(freight_aggregate, on="order_id", how="left")

In [ ]:
# checking if the columns were successfully added to the dataframe
merged.columns.tolist()

# **EDA of the merged data**

In [ ]:
#plotting the distribution of Customer Satisfaction
plt.figure(figsize=(10, 6))
sns.countplot(x="satisfaction_category", data=merged)
plt.title("Distribution of Customer Satisfaction")
plt.show()

In [ ]:
#plotting the distribution of Delay Satisfaction
plt.figure(figsize=(10, 6))
sns.countplot(x="delay_category", data=merged)
plt.title("Distribution of Delay Categories")
plt.show()

In [ ]:
#plotting the distribution of Total Order Price
plt.figure(figsize=(10,6))
sns.histplot(merged["total_price"], bins=100, kde=False)
plt.title("Distribution of Total Order Price")
plt.xlabel("Total Price")
plt.ylabel("Number of Orders")
plt.show()

In [ ]:
#plotting the distribution of Total Order Price
plt.figure(figsize=(10,6))
sns.histplot(merged["total_freight"], bins=100, kde=False)
plt.title("Distribution of Total Freight")
plt.xlabel("Total Freight")
plt.ylabel("Number of Orders")
plt.show()

In [ ]:
#plotting the Satisfaction Category by Delay Category
plt.figure(figsize=(10, 6))
sns.countplot(x="delay_category", hue="satisfaction_category", data=merged)
plt.title("Satisfaction Category by Delay Category")
plt.show()

In [ ]:
#find min and max of total price
print(merged["total_price"].min())
print(merged["total_price"].max())
# define bins and labels
total_price_bins = [0, 1500, 2500, 3500, 4500, 5500, merged["total_price"].max()]
labels = ["0-1500","1500-2500","2500-3500","3500-4500","4500-5500","5500+"]
#splitting to bins
merged["price_bin"] = pd.cut(merged["total_price"], bins=total_price_bins, labels=labels, include_lowest=True)
#plotting the Satisfaction by Total Price
plt.figure(figsize=(10,6))
sns.countplot(data=merged, x="price_bin", hue="satisfaction_category")
plt.title("Satisfaction by Total Price")
plt.xlabel("Order Total Price Range")
plt.ylabel("Number of Orders")
plt.legend(title="Satisfaction")
plt.show()

In [ ]:
#find min and max of total freight
print(merged["total_freight"].min())
print(merged["total_freight"].max())
# define bins and labels
total_freight_bins = [0, 150, 250, 350, 450, 550, merged["total_freight"].max()]
labels = ["0-150","150-250","250-350","350-450","450-550","550+"]
#splitting to bins
merged["freight_bin"] = pd.cut(merged["total_freight"], bins=total_freight_bins, labels=labels, include_lowest=True)
#plotting the Satisfaction by Total freight
plt.figure(figsize=(10,6))
sns.countplot(data=merged, x="freight_bin", hue="satisfaction_category")
plt.title("Satisfaction by Total Freight")
plt.xlabel("Order Total Freight Range")
plt.ylabel("Number of Orders")
plt.legend(title="Satisfaction")
plt.show()

In [ ]:
# Group orders  based on the number of items
bins = [0, 1, 3, 5, merged["num_items"].max()]
labels = ["1 item", "2–3 items", "4–5 items", "6+ items"]
merged["num_items_group"] = pd.cut(merged["num_items"], bins=bins, labels=labels, include_lowest=True)
#plotting the Satisfaction by Number of items in order
plt.figure(figsize=(10,6))
sns.countplot(data=merged, x="num_items_group", hue="satisfaction_category",
              order=labels, hue_order=["not satisfied","satisfied","extremely satisfied"])

plt.title("Satisfaction by Number of Items in Order")
plt.xlabel("Number of Items Group")
plt.ylabel("Number of Orders")
plt.legend(title="Satisfaction")
plt.show()

# **exporting the merged data to csv**

In [ ]:
merged.to_csv("merged.csv", index=False, date_format="%Y-%m-%d")

# **MapReduce**

In [ ]:
# export a smaller dataset with only the columns needed for mapreduce analysis
merged.to_csv(
    "mapreduce_dataset.csv",
    columns=[
        "order_id",
        "delay_category",
        "num_items_group",
        "price_bin",
        "freight_bin",
        "satisfaction_category"
    ],
    index=False
)

In [ ]:
%%writefile mapper.py
#!/usr/bin/env python3
import sys, csv, io

for line in sys.stdin:
    #reader
    r = next(csv.reader(io.StringIO(line)))
    # skip header
    if r and r[0].strip().lower() == "order_id":
        continue
    # get values with padding if needed
    order_id, delay, items, price, freight, satisfaction = (r + [""]*6)[:6]
    # Normalize satisfaction label where its all in lowercase and without spaces
    satisfaction = (satisfaction or "").strip().lower()
    # keep only 3 valid labels other than that save as "unknown"
    if satisfaction in ["not satisfied", "satisfied", "extremely satisfied"]:
        satisfaction_cat = satisfaction
    else:
        satisfaction_cat = "unknown"
    # emit 4 aspects delay, items, price, freight
    # each key is (aspect, value) and each value is
    print(f"Delay category: ,{delay or 'Unknown'}\t{satisfaction_cat},1")
    print(f"Items bin: ,{items or 'Unknown'}\t{satisfaction_cat},1")
    print(f"Price bin: ,{price or 'Unknown'}\t{satisfaction_cat},1")
    print(f"Freight bin: ,{freight or 'Unknown'}\t{satisfaction_cat},1")

In [ ]:
%%writefile reducer.py
#!/usr/bin/env python3
import sys
from collections import defaultdict
#for tracing teh current key that is being processed
currentK = None
# dictionary for holding the counts of the category for each key
counts = defaultdict(int)
# when a key is finshed processing result is prited
def flush(key, counts):
    result = ",".join([f"{k}:{v}" for k,v in counts.items()])
    print(f"{key}\t{result}")
# loop to process each line of mapper input
for line in sys.stdin:
    key, value = line.strip().split("\t", 1)
    satisfaction_cat, c = value.split(",", 1)
    c = int(c)
    #if we are processing the same key add 1
    if currentK == key:
        counts[satisfaction_cat] += c
    #if not flush the results of the previous key
    else:
        if currentK:
            flush(currentK, counts)
        currentK = key
        # reset counts for the new key
        counts = defaultdict(int)
        counts[satisfaction_cat] = c
#when exiting the loop flush the results of the previous key
if currentK:
    flush(currentK, counts)

In [ ]:
cat mapreduce_dataset.csv | python3 mapper.py | sort | python3 reducer.py
